# Connect to the Weaviate database

In [ ]:
import weaviate
client = weaviate.connect_to_local()
client.connect()

*Delete already existing collection if needed*

In [ ]:
# if client.collections.exists(collection_name):
#     client.collections.delete(collection_name)

In [ ]:
collection_name = "traffic_analyser"
query_file_dir = "/home/simon/new_outputs/JS-BM-P487__20250519-044003/2025_07/11/16/17/"
database_data_dir = "/home/simon/new_outputs/JS-BM-P498__20250519-050013/2025_07/12/10/36/"

# Database scheme (table) creation

In [ ]:
import weaviate.classes.config as wc

if not client.collections.exists(collection_name):
    collection = client.collections.create(
        name=collection_name,
        properties=[
            wc.Property(
                name="record_id",
                data_type=wc.DataType.INT
            ),
            wc.Property(
                name="video_id",
                data_type=wc.DataType.INT
            ),
            wc.Property(
                name="track_id",
                data_type=wc.DataType.INT
            ),
            wc.Property(
                name="frame_id",
                data_type=wc.DataType.INT
            ),
            wc.Property(
                name="position",
                data_type=wc.DataType.OBJECT,
                nested_properties=[
                    wc.Property(name="x", data_type=wc.DataType.INT),
                    wc.Property(name="y", data_type=wc.DataType.INT),
                ]
            ),
            wc.Property(
                name="detector_confidence",
                data_type=wc.DataType.NUMBER
            ),
            wc.Property(
                name="crop_path",
                data_type=wc.DataType.TEXT
            ),
        ],
        vector_config=wc.Configure.Vectors.self_provided(
            name="feature",
            vector_index_config=wc.Configure.VectorIndex.hnsw(
                distance_metric=wc.VectorDistances.COSINE,
                ef_construction=128,
                max_connections=64, # TODO: Find optimal value
                # Use quantizer=wc.Configure.VectorIndex.Quantizer.pq() for production
            )
        )
    )

# Data loading from existing extracted features

In [ ]:
from tools.embs_tools import get_all_records

all_records = get_all_records(database_data_dir)

string_to_coord_list = lambda s: [int(float(x)) for x in s[1:-2].split(",")]
string_to_coord_dict = lambda s: {"x": string_to_coord_list(s)[0], "y": string_to_coord_list(s)[1]}
all_meta = [
    {
        "record_id": int(record["record_id"]),
        "video_id": 0,
        "track_id": int(record["track_id"]),
        "frame_id": int(record["frame_id"]),
        "position": string_to_coord_dict(record["position"]),
        "detector_confidence": float(record["confidence"]),
        "crop_path": record["crop_path"],
    }
    for record in all_records
]

all_vectors = [
    record["embedding"].tolist()
    for record in all_records
]

# Inserting data to the database

In [ ]:
for meta, vec in zip(all_meta, all_vectors):
    collection.data.insert(
        properties=meta,
        vector=vec
    )

# Load query data from file and fetch database collection

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from tools.embs_tools import get_all_records, get_crops_for_id
from weaviate.classes.query import MetadataQuery
import random

collection = client.collections.get(collection_name)

all_records = get_all_records(query_file_dir)
random.shuffle(all_records)

# TOP-k matches visualizer

In [ ]:
import matplotlib.pyplot as plt
import cv2

num_queries = 5
top_k = 5

fig, axes = plt.subplots(num_queries, top_k + 1, figsize=(18, 3 * num_queries))
for query_idx in range(num_queries):
    query_record = all_records[query_idx]
    
    response = collection.query.near_vector(
        near_vector=query_record["embedding"],
        limit=top_k,
        return_metadata=MetadataQuery(distance=True),
    )
    
    query_crop = get_crops_for_id(
        query_file_dir,
        query_record["track_id"]
    )[-1]
    
    axes[query_idx, 0].imshow(cv2.cvtColor(query_crop, cv2.COLOR_BGR2RGB))
    axes[query_idx, 0].set_title(
        f'Query {query_idx + 1}\nTrack: {query_record["track_id"]}',
        fontsize=10,
        fontweight='bold'
    )
    axes[query_idx, 0].axis('off')
    axes[query_idx, 0].set_facecolor('#e8f4f8')
    
    for match_idx, obj in enumerate(response.objects):
        similarity_score = 1 - obj.metadata.distance
        match_crop = get_crops_for_id(database_data_dir, obj.properties["track_id"])[-1]
        
        axes[query_idx, match_idx + 1].imshow(cv2.cvtColor(match_crop, cv2.COLOR_BGR2RGB))
        axes[query_idx, match_idx + 1].set_title(
            f'Match {match_idx + 1}\nSim: {similarity_score:.3f}', 
            fontsize=9
        )
        axes[query_idx, match_idx + 1].axis('off')

plt.tight_layout()
plt.show()